# 04 · Ingesta e Índice Vectorial

Este notebook construye el índice vectorial de Aurum Market usando Qdrant.

Objetivos:
- Cargar el catálogo completo.
- Generar embeddings con el modelo seleccionado (E5-small).
- Crear colecciones Qdrant.
- Ingerir vectores con payloads.
- Verificar que los puntos se han insertado correctamente.
- Ejecutar las primeras búsquedas vectoriales.

Este notebook marca el inicio del sistema de búsqueda semántica.


#### Importar librerías y cargar datos

In [1]:
import sys
import os

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

print("Ruta añadida al PYTHONPATH:", ROOT_DIR)


Ruta añadida al PYTHONPATH: /home/alexd/modulo_vector_bbdd/actividad_evaluable


In [2]:
import pandas as pd
import numpy as np

from src.utils import safe_read_csv, log_section
from src.embeddings import embed_product, get_vector_dimension
from src.vector_ingestion import create_qdrant_collection, upsert_vector
from src.search_engine import search


#### Cargar catálogo completo

In [3]:
log_section("Cargar catálogo completo")

df_catalog = safe_read_csv("../data/catalogo_productos.csv.gz")
df_catalog.head()


[AURUM] 
[AURUM] ============================================================
[AURUM] Cargar catálogo completo
[AURUM] ============================================================
[AURUM] [CSV] Cargado: ../data/catalogo_productos.csv.gz (15000 filas)


,record_id,product_id,title,brand,color,locale,text,catalog_version,active
0,e1a0e559-6a49-5be5-b617-ec8a4899e975,B000G3T55M,NIKE Legasee Legging Swoosh Pantalones Deporti...,NIKE,Negro (Black/White 011),es,NIKE Legasee Legging Swoosh Pantalones Deporti...,1,True
1,0df8a596-0bc2-5deb-bc84-69b63972f975,B07NV4L2W5,"Interruptor Universal Inteligente con Wi-Fi, c...",meross,Blanco,es,"Interruptor Universal Inteligente con Wi-Fi, c...",1,True
2,ef061958-504a-505c-a7c8-0433be2d7630,B01BYFSX6M,TECKNET Mini Ratón Inalámbrico Wireless Mouse ...,TECKNET,Azul,es,TECKNET Mini Ratón Inalámbrico Wireless Mouse ...,1,True
3,d5e7e812-02e7-529f-97a3-e5a951d70e25,B005MWF7KO,Nostalgia,NaN,NaN,es,Nostalgia,1,True
4,f5726a4e-1d31-5c25-b109-2f3fb8aa6567,B00BEFAR80,Gel de contacto 250g. axion | Mejora la conduc...,axion,NaN,es,Gel de contacto 250g. axion | Mejora la conduc...,1,True


#### Seleccionar modelo de embeddings

In [4]:
log_section("Seleccionar modelo")

model_name = "e5_small"
vector_dim = get_vector_dimension(model_name)

print("Modelo:", model_name)
print("Dimensión del vector:", vector_dim)


[AURUM] 
[AURUM] ============================================================
[AURUM] Seleccionar modelo
[AURUM] ============================================================


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[AURUM] [EMBEDDINGS] Modelo cargado: e5_small (intfloat/multilingual-e5-small)
Modelo: e5_small
Dimensión del vector: 384


/home/alexd/modulo_vector_bbdd/actividad_evaluable/src/embeddings.py:81: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return model.get_sentence_embedding_dimension()


#### Crear colección Qdrant

In [5]:
log_section("Crear colección Qdrant")

create_qdrant_collection(
    model_name=model_name,
    vector_dim=vector_dim,
    metric="dot"  # E5-small usa dot product
)

[AURUM] 
[AURUM] ============================================================
[AURUM] Crear colección Qdrant
[AURUM] ============================================================
[AURUM] [QDRANT] Colección creada: aurum_e5_small (dim=384, metric=dot)


#### Generar embeddings y payloads

In [6]:
log_section("Generar embeddings y payloads")

def build_payload(row):
    return {
        "record_id": row["record_id"],
        "product_id": row["product_id"],
        "title": row["title"],
        "brand": row["brand"],
        "color": row["color"],
        "active": int(row["active"]),
        "catalog_version": int(row["catalog_version"])
    }

# Ingesta completa para el recorrido final.
subset = df_catalog

vectors = []
payloads = []

for _, row in subset.iterrows():
    text = row["title"] + " " + row["text"]
    vec = embed_product(text, model_name=model_name)
    vectors.append(vec)
    payloads.append(build_payload(row))

len(vectors)

[AURUM] 
[AURUM] ============================================================
[AURUM] Generar embeddings y payloads
[AURUM] ============================================================


15000

#### Ingesta vectorial en Qdrant

In [7]:
log_section("Ingesta vectorial en Qdrant")

for i, row in subset.iterrows():
    record_id = row["record_id"]
    vec = vectors[i]
    payload = payloads[i]

    upsert_vector(model_name, record_id, vec, payload)

print("Ingesta completada.")


[AURUM] 
[AURUM] ============================================================
[AURUM] Ingesta vectorial en Qdrant
[AURUM] ============================================================
Ingesta completada.


#### Verificación de puntos en Qdrant

In [8]:
log_section("Verificación de puntos")

# Elegimos un record_id aleatorio
sample_id = subset.iloc[10]["record_id"]
sample_title = subset.iloc[10]["title"]

print("Record ID:", sample_id)
print("Título:", sample_title)

results = search(sample_title, top_k=5, model_name=model_name)
results


[AURUM] 
[AURUM] ============================================================
[AURUM] Verificación de puntos
[AURUM] ============================================================
Record ID: 366b1ec8-fb58-5ef6-b299-cfbe98b9b3b0
Título: Onforu 10M Tira LED Regulable, 3000K Blanca Cálida Tira de Luces, 12V Cinta led Interior con Adaptador, 2400LM Tira LED 2835, Decoración Led para Habitación, Sala, Gabinete, Pared, Armario, Escalera


[{'rank': 1,
  'record_id': '366b1ec8-fb58-5ef6-b299-cfbe98b9b3b0',
  'product_id': 'B07C3Y2Z1W',
  'title': 'Onforu 10M Tira LED Regulable, 3000K Blanca Cálida Tira de Luces, 12V Cinta led Interior con Adaptador, 2400LM Tira LED 2835, Decoración Led para Habitación, Sala, Gabinete, Pared, Armario, Escalera',
  'brand': 'Onforu',
  'color': 'Blanco Cálido',
  'active': 1,
  'score': 0.9141233},
 {'rank': 2,
  'record_id': '0735075b-af51-5efa-b49f-a8ccbdeba140',
  'product_id': 'B07TJXZNDZ',
  'title': 'LE LED Luces de Tiras Regulables, 5M 1200lm, Blanco Frío 6000K, 300 LEDs, Enchufe en la tira de luz para gabinete, armario y más, Incluido Fuente de alimentación de 12V y regulador de intensidad',
  'brand': 'Lighting EVER',
  'color': 'Blanco Frío Regulable',
  'active': 1,
  'score': 0.89269495},
 {'rank': 3,
  'record_id': '001a2c86-16b4-5c95-9709-a7a09832baf5',
  'product_id': 'B089GDHKRY',
  'title': 'Lepro Tira de LED 10M blanca cálida 3000K, Tira luz regulable 600 LED SMD 2835, ca

#### Primera búsqueda vectorial real

In [9]:
log_section("Primera búsqueda vectorial")

query = "zapatillas running hombre"
results = search(query, top_k=10, model_name=model_name)

pd.DataFrame(results)


[AURUM] 
[AURUM] ============================================================
[AURUM] Primera búsqueda vectorial
[AURUM] ============================================================


,rank,record_id,product_id,title,brand,color,active,score
0,1,934d64df-7895-5e72-9114-c3ffc274030c,B08ZXMXS8J,Zapatillas Casual Hombre Running Zapatos Moda ...,SANNAX,Azul,1,0.908013
1,2,cef077ed-319b-5f9a-982a-3328424a72f5,B07K6ZK1T8,"Asics Patriot 10, Zapatillas de Running Hombre...",ASICS,Azul Imperial White 402,1,0.904066
2,3,fa8844be-206a-55ce-9524-88dc9c2a1e96,B078RRGLT6,"Nike Revolution 4 EU, Zapatillas de Running pa...",NIKE,White White Pure Platinum,1,0.900440
3,4,45df87eb-89d1-58b3-87a9-47dfae5f3e5a,B08XWJ3YW9,Zapatillas De Deporte Hombres Running Correr T...,ZMBCYG,Blanco,1,0.900316
4,5,a1bb9604-3b79-5280-9be6-261b15189d81,B078RSZ2MR,"Nike Revolution 4 EU, Zapatillas de Running pa...",NIKE,White White Pure Platinum,1,0.899834
5,6,7f755b35-b2ae-5669-aba0-724663ac91e6,B0743GLWXC,"Reebok Run Supreme 3.0, Zapatillas de Running ...",Reebok,Azul Collegiate Navy Smoky Indigo Pewter White,1,0.899103
6,7,fec81ac1-228c-5d57-9689-1fd7ea99f35c,B07YV2XK1C,BRONAX Zapatillas Hombres Deporte Running Zapa...,BRONAX,High Top Tudo Blanco,1,0.892884
7,8,27c41e05-bd78-549a-b776-177b3672b497,B07LG17SHH,"adidas Terrex Swift R2 GTX, Zapatillas de Runn...",adidas,Gris Grey Core Black Grey 0,1,0.892805
8,9,59aeeb28-24ca-5beb-8a14-2b8d0f571adf,B0059KZJHU,"Nike Flyknit Racer, Zapatillas de Running Homb...",NIKE,Blanco Black White,1,0.890954
9,10,95dff8f8-d4e6-558d-b766-d0a0336d6f4b,B07RMB8X26,"Nike Air MAX Command, Zapatillas de Running Ho...",NIKE,Gris Pure Platinum Gym Red Dk Grey Cool Grey W...,1,0.890762


#### Interpretación de resultados

La búsqueda vectorial devuelve productos semánticamente relacionados:

- Zapatillas de running.
- Calzado deportivo.
- Modelos de hombre.
- Variantes de color y marca coherentes.

Observaciones:
- El motor vectorial captura sinónimos y variaciones de lenguaje.
- La relevancia es mucho mayor que en BM25.
- Los resultados son coherentes incluso con descripciones largas.

Este notebook demuestra que el índice vectorial está correctamente construido.


### Conclusiones

- El catálogo se ha cargado correctamente.
- Se ha creado la colección Qdrant para E5-small.
- Se han generado embeddings y payloads.
- Se han ingerido vectores en Qdrant sin errores.
- Las primeras búsquedas vectoriales son coherentes y relevantes.

En el siguiente notebook aplicaremos **filtros vectoriales** y realizaremos búsquedas avanzadas.
